In [ ]:
from langchain_ollama import ChatOllama
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_community.document_loaders import PyPDFLoader
import pandas as pd
from langchain_chroma import Chroma
import torch
# print(torch.__version__, torch.version.cuda)
# print(torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

CUDA available: True


In [ ]:
#importing dataset
df = pd.read_csv("../datasets/pubmed abstracts/pubmed_abstracts copy.csv")
df.head()



,Unnamed: 0,deep_learning
0,0,(['Magnetic resonance spectroscopic imaging (M...
1,1,(['Existing deep convolutional neural networks...
2,2,(['Deep learning techniques have been increasi...
3,3,(['The original article unfortunately containe...
4,4,(['The most common applications of artificial ...


In [3]:
#loading dataset in langchain

docs = []
for col in df.columns:
    if col == "Unnamed: 0":
        continue
    text = "\n".join(df[col].dropna().astype(str))
    docs.append(Document(page_content=text, metadata={"column": col}))

print(len(docs))   # = number of columns
print(docs[0].metadata)
print(type(docs[0].page_content))

1
{'column': 'deep_learning'}
<class 'str'>


In [4]:
#Spliting text

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
texts = text_splitter.split_documents(docs)
print(len(texts)) 
print(texts[5].page_content)

22842
by the classification branch. Feeding a chest radiograph to the trained Thorax-Net, a diagnosis is obtained by averaging and binarizing the outputs of two branches. The proposed Thorax-Net model has been evaluated against three state-of-the-art deep learning models using the patientwise official split of the ChestX-ray14 dataset and against other five deep learning models using the imagewise random data split. Our results show that Thorax-Net achieves an average per-class area under the receiver operating characteristic curve (AUC) of 0.7876 and 0.896 in both experiments, respectively, which are higher than the AUC values obtained by other deep models when they were all trained with no external data.'], 'Thorax-Net: An Attention Regularized Deep Neural Network for Classification of Thoracic Diseases on Chest Radiography.')


In [5]:
#Embeddings Model 

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
    encode_kwargs={"batch_size": 512},
    )

embeddings._client.half()                 # fp16
embeddings._client.max_seq_length = 256   # truncate long abstracts

query_embedding = embeddings.embed_query("What is a sentence embedding?")
doc_embeddings = embeddings.embed_documents(
    [
        "Sentence embeddings map text to dense vectors.",
        "LangChain provides a standard Embeddings interface.",
    ]
)

print(type(embeddings))
print(hasattr(embeddings, "_client"), hasattr(embeddings, "client"))

s:\Projects\rag-dissertation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 701.89it/s]


<class 'langchain_huggingface.embeddings.huggingface.HuggingFaceEmbeddings'>
True False


In [7]:
m = embeddings._client
print(m.max_seq_length)                  # 256
print(next(m.parameters()).dtype)        # torch.float16
print(next(m.parameters()).device)       # cuda:0

256
torch.float16
cuda:0


In [ ]:
# Embeddings & vector storage
#
# This cell only READS the Chroma DB that was persisted to ./chroma_langchain_db
# (src/chroma_langchain_db) -- it does not re-embed the dataset on every run.

from pathlib import Path

CHROMA_DIR = Path("chroma_langchain_db")
if not CHROMA_DIR.exists():
    raise FileNotFoundError(
        f"Local Chroma DB not found at {CHROMA_DIR.resolve()}. "
        "Run the ingestion cells once with REBUILD_INDEX = True to create it."
    )

vector_store = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings,
    persist_directory=str(CHROMA_DIR),
)

# Default LangChain retriever over the persisted collection.
retriever = vector_store.as_retriever()

print(f"Loaded {vector_store._collection.count()} chunks from {CHROMA_DIR.resolve()}")


In [ ]:
# Timeline of storage
#
# The index already exists on disk, so ingestion is opt-in: flip REBUILD_INDEX to
# True only when you want to (re)embed the split texts into the collection.

from tqdm import tqdm

REBUILD_INDEX = False

if REBUILD_INDEX:
    max_bs = vector_store._client.get_max_batch_size()   # 5461 for you
    BATCH = min(2000, max_bs)                            # stay safely under it

    for i in tqdm(range(0, len(texts), BATCH)):
        batch = texts[i:i + BATCH]
        ids = [str(d.metadata.get("pmid", i + j)) for j, d in enumerate(batch)]
        vector_store.add_documents(batch, ids=ids)
else:
    print("Skipping ingestion; reading the existing local Chroma DB as-is.")


In [24]:
# Retrieve evidence for the same question that will be evaluated.
query = "What was the accuracy of dual subpace study?"
retrieved_docs = retriever.invoke(query)
retrieval_context = [doc.page_content for doc in retrieved_docs]
if not retrieval_context:
    raise ValueError("The retriever returned no context; cannot evaluate a RAG answer.")
print(f"Retrieved {len(retrieval_context)} context chunks.")

# Show the retrieved chunks (rank, metadata, text) so the evidence is visible.
for rank, doc in enumerate(retrieved_docs, start=1):
    meta = ", ".join(f"{k}={v}" for k, v in doc.metadata.items()) or "none"
    print(f"--- chunk {rank}/{len(retrieved_docs)} [{meta}] ---")
    print(doc.page_content)
    print()


Retrieved 4 context chunks.
--- chunk 1/4 [column=deep_learning] ---
CWK\u2009=\u20090.886 ± 0.004 (95% confidence interval [CI] 0.879-0.894), S\u2009=\u20090.967%, SP\u2009=\u20090.976%, PPV\u2009=\u20090.836%, and NPV\u2009=\u20090.996%. The error type I\u2009=\u20090.024, and the error type II\u2009=\u20090.004. Likewise, the referable-DR results were: CWK\u2009=\u20090.809 (95% CI 0.798-0.819), S\u2009=\u20090.998, SP\u2009=\u20090.968, PPV\u2009=\u20090.701, NPV\u2009=\u20090.928, error type I\u2009=\u20090.032, and error type II\u2009=\u20090.001. Discussion: Our DLA can be used as a high confidence diagnostic tool to help in DR screening, especially when it might be difficult for ophthalmologists or other professionals to identify. It can identify patients with any-DR and those that should be referred. Conclusions: The DLA can be valid to aid in screening of DR."], 'Validation of a Deep Learning Algorithm for Diabetic Retinopathy.')

--- chunk 2/4 [column=deep_learning] ---
or a

In [13]:
#Setting up LLM

llm = ChatOllama(
    model="qwen3.5:2b", 
    temperature = 0.5,
    reasoning=False,
    
)

In [25]:
# Generate the RAG answer from the retrieved evidence.
context_text = "\n\n".join(retrieval_context)
prompt = f"""Answer the question using only the supplied context. If the context does not contain the answer, say so.

Context:
{context_text}

Question: {query}
Answer:"""
response = llm.invoke(prompt)
print(response.content)


Based on the provided context, there is no mention of a "dual subpace study" or its accuracy. The context discusses diagnostic tools (DLA), diabetic retinopathy screening results (CWK, S, SP, PPV, NPV), and models for deep learning in radiology and particle physics.

Answer: Not mentioned in the context.


In [ ]:
# Sanity check the local Ollama server with the judge model.
import ollama

r = ollama.chat(
    model="qwen3.5:2b",
    messages=[{"role": "user", "content": 'Say hi in JSON: {"msg": ...}'}],
)
print(repr(r.message.content))


In [26]:
import os

# The local judge can be slow, so allow longer local judge calls.
os.environ["DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE"] = "300"
os.environ["DEEPEVAL_PER_TASK_TIMEOUT_SECONDS_OVERRIDE"] = "600"

actual_context =[
    "As one of the most common neurobehavioral diseases in school-age children, Attention Deficit Hyperactivity Disorder (ADHD) has been increasingly studied in recent years. But it is still a challenge problem to accurately identify ADHD patients from healthy persons. To address this issue, we propose a dual subspace classification algorithm by using individual resting-state Functional Connectivity (FC). In detail, two subspaces respectively containing ADHD and healthy control features, called as dual subspaces, are learned with several subspace measures, wherein a modified graph embedding measure is employed to enhance the intra-class relationship of these features. Therefore, given a subject (used as test data) with its FCs, the basic classification principle is to compare its projected component energy of FCs on each subspace and then predict the ADHD or control label according to the subspace with larger energy. However, this principle in practice works with low efficiency, since the dual subspaces are unstably obtained from ADHD databases of small size. Thereby, we present an ADHD classification framework by a binary hypothesis testing of test data. Here, the FCs of test data with its ADHD or control label hypothesis are employed in the discriminative FC selection of training data to promote the stability of dual subspaces. For each hypothesis, the dual subspaces are learned from the selected FCs of training data. The total projected energy of these FCs is also calculated on the subspaces. Sequentially, the energy comparison is carried out under the binary hypotheses. The ADHD or control label is finally predicted for test data with the hypothesis of larger total energy. In the experiments on ADHD-200 dataset, our method achieves a significant classification performance compared with several state-of-the-art machine learning and deep learning methods, where our accuracy is about 90 \% for most of ADHD databases in the leave-one-out cross-validation test.'], 'ADHD classification by dual subspace learning using resting-state functional connectivity."
]

from deepeval.metrics import (
    AnswerRelevancyMetric,
    FaithfulnessMetric,
    HallucinationMetric,
)
from deepeval.models.llms.ollama_model import OllamaModel, retry_ollama
from deepeval.test_case import Audio, LLMTestCase
from pydantic import BaseModel
from typing import List, Optional, Tuple, Union


class NonThinkingOllamaModel(OllamaModel):
    @retry_ollama
    def generate(
        self,
        prompt: str,
        schema: Optional[BaseModel] = None,
        audio: Optional[List[Audio]] = None,
    ) -> Tuple[Union[str, BaseModel], float]:
        response = self.load_model().chat(
            model=self.name,
            messages=self._prompt_messages(prompt, audio),
            think=False,
            format=schema.model_json_schema() if schema else None,
            options={
                "temperature": self.temperature,
                **self.generation_kwargs,
            },
        )
        content = response.message.content
        if not content:
            raise ValueError(
                "Ollama returned empty content for the DeepEval judge response."
            )
        result = schema.model_validate_json(content) if schema else content
        return result, 0

# Use a separate local judge; Qwen 3.5 otherwise returns reasoning instead of JSON.
evaluation_model = NonThinkingOllamaModel(
    model="qwen3.5:2b",
    base_url="http://localhost:11434",
    temperature=0,
)

test_case = LLMTestCase(
    input=query,
    actual_output=response.content,
    context=actual_context,
    retrieval_context=retrieval_context,
)

metrics = {
    "hallucination": HallucinationMetric(
        threshold=0.5, model=evaluation_model, include_reason=True, async_mode=False
    ),
    "faithfulness": FaithfulnessMetric(
        threshold=0.5, model=evaluation_model, include_reason=True, async_mode=False
    ),
    "answer relevancy": AnswerRelevancyMetric(
        threshold=0.5, model=evaluation_model, include_reason=True, async_mode=False
    ),
}

for name, metric in metrics.items():
    metric.measure(test_case)
    print(f"{name}: {metric.score:.3f}")
    print(f"  passed: {metric.is_successful()}")
    print(f"  reason: {metric.reason}")


s:\Projects\rag-dissertation\.venv\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

hallucination: 0.000
  passed: False
  reason: The hallucination score is 1.0 because the actual output completely contradicts the provided context, which explicitly details a specific dual subspace classification algorithm with 90% accuracy on the ADHD-200 dataset; the absence of this study in the output constitutes a direct factual error rather than a minor inconsistency.


faithfulness: 1.000
  passed: True
  reason: The faithfulness score is 1.00 because there are no contradictions in the retrieval context, and the actual output aligns perfectly with the provided information.


answer relevancy: 0.000
  passed: False
  reason: The score is 0.00 because both statements explicitly contradict the input by stating that no mention exists for a 'dual subpace study', which directly addresses the user's query about its accuracy.
